# ASAP8 raw SLAP2 scan movie

Reconstruct the original cycle-by-cycle SLAP2 sampling pattern around one soma:

- the raw multi-ROI raster page comes from each cycle `.tif`;
- the high-rate integration samples come directly from the paired `.dat`;
- `pixelReplacementMaps` expands each integration superpixel over its true DMD footprint;
- the result is overlaid on the static reference stack and saved as an MP4.

One movie frame corresponds to one 222-line scan cycle (about 20.8 ms in this
session). No extracted ROI trace is used. `VALUE_MODE="dff"` only applies a
per-pixel median baseline to make voltage-dependent changes visible; use
`VALUE_MODE="raw"` to show absolute detector values.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd
import shutil
import tifffile
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from scipy import sparse
from scipy.io import loadmat
from IPython.display import Video, display

ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path is None:
    import imageio_ffmpeg
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

mpl.rcParams["animation.ffmpeg_path"] = ffmpeg_path

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

## Session and movie settings

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MOUSE = 852835
SESSION_IDX = -3

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

DMD = 1
ACQUISITION_INDEX = 0
ROI_INDEX = 0             # zero-based parse-plan ROI ID
RAW_CHANNEL = 1
REFERENCE_CHANNEL = 1

T_START_SEC = 10.0
T_STOP_SEC = 20.0
CROP_SIZE_PX = 250

VALUE_MODE = "dff"        # "dff" or "raw"
PLAYBACK_RATE = 0.5      # 1 = real time; 0.5 = half speed
OVERLAY_ALPHA = 0.85

registry = VIPSessionRegistry.from_basepath(BASE_PATH)
session_df = registry.sessions(
    subject_ids=[TARGET_MOUSE],
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).sort_values("session_date").reset_index(drop=True)

assets = [registry.resolve_assets(row) for _, row in session_df.iterrows()]
asset = assets[SESSION_IDX]

display(session_df[[c for c in [
    "session_id", "subject_id", "session_date", "session_type",
    "dmd1_depth", "dmd2_depth", "quality",
] if c in session_df.columns]])
print("Selected:", asset.session_id)

## Locate the raw acquisition

In [ ]:
session_dir = Path(asset.session_dir)
trial_table_path = next(session_dir.rglob("trialTable.mat"))
dynamic_dir = next(
    p for p in session_dir.rglob("dynamic_data")
    if p.parent.name.lower() == "slap2"
)

meta_files = sorted(dynamic_dir.glob(f"*DMD{DMD}.meta"))
meta_path = meta_files[ACQUISITION_INDEX]
acquisition_stem = meta_path.stem

cycle_number = lambda p: int(re.search(r"CYCLE-(\d+)", p.name).group(1))
dat_files = sorted(
    dynamic_dir.glob(f"{acquisition_stem}-TRIAL*-CYCLE-*.dat"),
    key=cycle_number,
)
tif_files = [p.with_suffix(".tif") for p in dat_files]

print("Trial table:", trial_table_path)
print("Metadata:   ", meta_path)
print("Cycle files:", len(dat_files))
print("First DAT:  ", dat_files[0].name)
print("First TIFF: ", tif_files[0].name)

## Read the parse plan

The `.meta` file describes the spatial footprint and acquisition-line location
of every raw sample. Raster samples map one-to-one to pixels; integration
samples map many DMD pixels onto one superpixel ID.

In [ ]:
with h5py.File(meta_path, "r") as f:
    pp = f["AcquisitionContainer/ParsePlan"]
    plan = pp["acqParsePlan"]

    line_rate_hz = float(pp["lineRateHz"][0, 0])
    lines_per_cycle = int(pp["linesPerCycle"][0, 0])
    n_rows = int(f["dmdPixelsPerColumn"][0, 0])
    n_cols = int(f["dmdPixelsPerRow"][0, 0])
    saved_channels = f["channelsSave"][()].astype(int).ravel()
    raster_offset_xy = pp["rasterOffsetXY"][()].astype(int).ravel()
    raster_size_xy = pp["rasterSizeXY"][()].astype(int).ravel()
    z_um = float(f["remoteFocusPosition_um"][0, 0])

    line_superpixels = []
    line_roi_ids = []

    for line in range(lines_per_cycle):
        superpixels = np.asarray(
            f[plan["superPixelID"][line, 0]][()]
        ).ravel()
        roi_ids = np.asarray(
            f[plan["roiIds"][line, 0]][()]
        ).ravel()

        # The first parse-plan entry is a timing line with no detector samples.
        if line == 0 and np.array_equal(superpixels, [0, 1]):
            superpixels = np.array([], dtype=np.uint32)
            roi_ids = np.array([], dtype=np.int16)

        line_superpixels.append(superpixels)
        line_roi_ids.append(roi_ids)

    actual_pixel, replacement_id = np.asarray(
        f[pp["pixelReplacementMaps"][0, 0]][()]
    )

    roi_refs = f["AcquisitionContainer/ROIs/rois"]
    roi_shape = np.asarray(
        f[roi_refs[ROI_INDEX, 0]]["shapeData"][()]
    ).T.astype(int) - 1

n_per_line = np.array([len(x) for x in line_superpixels])
line_byte_offset = np.r_[0, np.cumsum(40 + 4 * n_per_line[:-1])]

sample_superpixel = np.concatenate(line_superpixels)
sample_roi_id = np.concatenate(line_roi_ids)
sample_line = np.repeat(np.arange(lines_per_cycle), n_per_line)
sample_position = np.concatenate([np.arange(n) for n in n_per_line])

roi_table = []
for roi_id in sorted(set(sample_roi_id[sample_roi_id >= 0])):
    roi_table.append({
        "roi_id": int(roi_id),
        "samples_per_cycle": int(np.sum(sample_roi_id == roi_id)),
        "target_rate_hz": line_rate_hz * np.mean([
            np.any(ids == roi_id) for ids in line_roi_ids
        ]),
    })

display(pd.DataFrame(roi_table))
print(f"Line rate: {line_rate_hz:,.3f} Hz")
print(f"Cycle rate: {line_rate_hz / lines_per_cycle:.3f} Hz")
print("Saved channels:", saved_channels)

## Load the matching reference plane and define the crop

In [ ]:
trial_table = loadmat(
    trial_table_path,
    squeeze_me=True,
    struct_as_record=False,
)["trialTable"]

ref_stack = trial_table.refStack[DMD - 1]
z_index = int(np.argmin(np.abs(ref_stack.Zs - z_um)))
channel_index = int(np.where(ref_stack.channels == REFERENCE_CHANNEL)[0][0])
page_index = z_index * len(ref_stack.channels) + channel_index

reference = np.asarray(ref_stack.IM[:, :, page_index]).T

cy, cx = np.round(roi_shape.mean(axis=0)).astype(int)
crop_size = min(CROP_SIZE_PX, n_rows, n_cols)
half = crop_size // 2
y0 = int(np.clip(cy - half, 0, n_rows - crop_size))
x0 = int(np.clip(cx - half, 0, n_cols - crop_size))
y1, x1 = y0 + crop_size, x0 + crop_size

reference_crop = reference[y0:y1, x0:x1]
roi_mask = np.zeros((n_rows, n_cols), dtype=bool)
roi_mask[roi_shape[:, 0], roi_shape[:, 1]] = True
roi_mask_crop = roi_mask[y0:y1, x0:x1]

vmin, vmax = np.percentile(reference_crop, [1, 99.7])

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(reference_crop, cmap="gray", vmin=vmin, vmax=vmax)
ax.contour(roi_mask_crop, levels=[0.5], colors="white", linewidths=1.5)
ax.set_title(f"{asset.session_id} · DMD{DMD} ROI {ROI_INDEX}")
ax.axis("off")
plt.show()

## Map integration samples into the crop

The replacement map uses zero-based, row-major DMD pixel indices. The sparse
matrix below expands each selected superpixel measurement over all pixels in its
true footprint and averages repeated measurements within a scan cycle.

In [ ]:
pixel_y = actual_pixel // n_cols
pixel_x = actual_pixel % n_cols

inside_crop = (
    (pixel_y >= y0) & (pixel_y < y1) &
    (pixel_x >= x0) & (pixel_x < x1)
)

crop_actual_pixel = actual_pixel[inside_crop]
crop_replacement_id = replacement_id[inside_crop]

integration_sample = sample_roi_id == ROI_INDEX
sample_superpixel_roi = sample_superpixel[integration_sample]
sample_line_roi = sample_line[integration_sample]
sample_position_roi = sample_position[integration_sample]

rows = []
cols = []

for superpixel in np.unique(sample_superpixel_roi):
    pixels = crop_actual_pixel[crop_replacement_id == superpixel]
    pixel_flat = (
        (pixels // n_cols - y0) * crop_size +
        (pixels % n_cols - x0)
    )
    sample_cols = np.where(sample_superpixel_roi == superpixel)[0]

    rows.append(np.tile(pixel_flat, len(sample_cols)))
    cols.append(np.repeat(sample_cols, len(pixel_flat)))

rows = np.concatenate(rows)
cols = np.concatenate(cols)

footprint_matrix = sparse.csr_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(crop_size * crop_size, len(sample_superpixel_roi)),
)
footprint_weight = np.asarray(footprint_matrix.sum(axis=1)).ravel()

print("Integration samples per cycle:", len(sample_superpixel_roi))
print("Superpixels:", len(np.unique(sample_superpixel_roi)))
print("Footprint pixels:", np.sum(footprint_weight > 0))

## Read the requested raw `.dat` cycles

In [ ]:
header = np.fromfile(dat_files[0], dtype="<u4", count=34)
header_bytes = int(header[2])
header_values = dict(header[3:-1].reshape(-1, 2))
bytes_per_cycle = int(header_values[3])

cycle_period_sec = lines_per_cycle / line_rate_hz
first_cycle = int(np.floor(T_START_SEC / cycle_period_sec))
last_cycle = int(np.ceil(T_STOP_SEC / cycle_period_sec))
global_cycles = np.arange(first_cycle, last_cycle)

raw_channel_index = int(np.where(saved_channels == RAW_CHANNEL)[0][0])
sample_u16_index = (
    header_bytes
    + line_byte_offset[sample_line_roi]
    + 40
    + 2 * (
        raw_channel_index * n_per_line[sample_line_roi]
        + sample_position_roi
    )
) // 2

raw_integration = np.empty(
    (len(global_cycles), len(sample_superpixel_roi)),
    dtype=np.uint16,
)
filled = np.zeros(len(global_cycles), dtype=bool)

for dat_path in dat_files:
    file_first_cycle = cycle_number(dat_path)
    n_file_cycles = (
        dat_path.stat().st_size - header_bytes
    ) // bytes_per_cycle

    use = (
        (global_cycles >= file_first_cycle) &
        (global_cycles < file_first_cycle + n_file_cycles)
    )
    if not np.any(use):
        continue

    local_cycles = global_cycles[use] - file_first_cycle
    raw_u16 = np.memmap(dat_path, dtype="<u2", mode="r")

    raw_integration[use] = raw_u16[
        local_cycles[:, None] * (bytes_per_cycle // 2)
        + sample_u16_index[None, :]
    ]
    filled[use] = True

assert filled.all(), "The requested interval extends beyond the available cycle files."

frame_times = (global_cycles + 0.5) * cycle_period_sec
print("Frames:", len(frame_times))
print("Interval:", frame_times[0], "to", frame_times[-1], "s")

## Read only the matching TIFF pages

The TIFF may store a full DMD page or only the cropped raster region. This cell
reads one requested page at a time and keeps only the movie crop in memory.

In [ ]:
raster_frames = np.full(
    (len(global_cycles), crop_size, crop_size),
    np.nan,
    dtype=np.float32,
)

raster_x0, raster_y0 = raster_offset_xy
raster_width, raster_height = raster_size_xy

for tif_path in tif_files:
    file_first_cycle = cycle_number(tif_path)
    dat_path = tif_path.with_suffix(".dat")
    n_file_cycles = (
        dat_path.stat().st_size - header_bytes
    ) // bytes_per_cycle

    use_indices = np.where(
        (global_cycles >= file_first_cycle) &
        (global_cycles < file_first_cycle + n_file_cycles)
    )[0]
    if len(use_indices) == 0:
        continue

    local_cycles = global_cycles[use_indices] - file_first_cycle

    with tifffile.TiffFile(tif_path) as tif:
        n_pages = len(tif.pages)
        pages_are_interleaved = n_pages == n_file_cycles * len(saved_channels)

        for output_index, local_cycle in zip(use_indices, local_cycles):
            page_index = (
                int(local_cycle) * len(saved_channels) + raw_channel_index
                if pages_are_interleaved
                else int(local_cycle)
            )
            page = np.squeeze(tif.pages[page_index].asarray())

            if page.ndim == 3:
                page = (
                    page[..., raw_channel_index]
                    if page.shape[-1] == len(saved_channels)
                    else page[raw_channel_index]
                )

            if page.shape == (n_cols, n_rows):
                page = page.T

            if page.shape == (n_rows, n_cols):
                raster_frames[output_index] = page[y0:y1, x0:x1]
                continue

            if page.shape == (raster_width, raster_height):
                page = page.T

            ya = max(y0, raster_y0)
            yb = min(y1, raster_y0 + raster_height)
            xa = max(x0, raster_x0)
            xb = min(x1, raster_x0 + raster_width)

            raster_frames[
                output_index,
                ya - y0:yb - y0,
                xa - x0:xb - x0,
            ] = page[
                ya - raster_y0:yb - raster_y0,
                xa - raster_x0:xb - raster_x0,
            ]

print("Raster movie:", raster_frames.shape)

## Reconstruct each integration frame and combine it with the raster movie

In [ ]:
integration_frames = (
    footprint_matrix @ raw_integration.T
).T

integration_frames = np.divide(
    integration_frames,
    footprint_weight[None, :],
    out=np.full_like(integration_frames, np.nan),
    where=footprint_weight[None, :] > 0,
).reshape(-1, crop_size, crop_size)

if VALUE_MODE == "dff":
    raster_f0 = np.nanmedian(raster_frames, axis=0)
    integration_f0 = np.nanmedian(integration_frames, axis=0)

    raster_movie = np.divide(
        raster_frames - raster_f0,
        raster_f0,
        out=np.full_like(raster_frames, np.nan),
        where=raster_f0 > 0,
    )
    integration_movie = np.divide(
        integration_frames - integration_f0,
        integration_f0,
        out=np.full_like(integration_frames, np.nan),
        where=integration_f0 > 0,
    )
else:
    raster_movie = raster_frames
    integration_movie = integration_frames

movie = raster_movie.copy()
replace = np.isfinite(integration_movie)
movie[replace] = integration_movie[replace]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(
    reference_crop,
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title("Reference")

axes[1].imshow(raster_movie[0], cmap="gray")
axes[1].set_title("Raw raster cycle")

if VALUE_MODE == "dff":
    limit = np.nanpercentile(np.abs(integration_movie), 99.5)
    axes[2].imshow(
        integration_movie[0],
        cmap="coolwarm",
        vmin=-limit,
        vmax=limit,
    )
else:
    axes[2].imshow(
        integration_movie[0],
        cmap="inferno",
        vmin=np.nanpercentile(integration_movie, 1),
        vmax=np.nanpercentile(integration_movie, 99.5),
    )
axes[2].set_title("Raw integration reconstruction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Render the MP4

In [ ]:
output_dir = Path(asset.derived_dir) / "voltage" / "movies"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / (
    f"{asset.session_id}_DMD{DMD}_ROI{ROI_INDEX}_"
    f"raw_scan_{T_START_SEC:g}-{T_STOP_SEC:g}s_{VALUE_MODE}.mp4"
)

fig, ax = plt.subplots(figsize=(7.2, 7.2), dpi=100)
ax.imshow(
    reference_crop,
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)

if VALUE_MODE == "dff":
    limit = np.nanpercentile(np.abs(movie), 99.5)
    overlay = ax.imshow(
        movie[0],
        cmap="cool_r",
        vmin=-limit,
        vmax=limit,
        alpha=OVERLAY_ALPHA,
    )
else:
    raw_limits = np.nanpercentile(movie, [1, 99.5])
    overlay = ax.imshow(
        movie[0],
        cmap="inferno",
        vmin=raw_limits[0],
        vmax=raw_limits[1],
        alpha=OVERLAY_ALPHA,
    )

ax.contour(
    roi_mask_crop,
    levels=[0.5],
    colors="white",
    linewidths=1.5,
)

time_text = ax.text(
    0.98,
    0.98,
    "",
    transform=ax.transAxes,
    ha="right",
    va="top",
    color="white",
    fontsize=36,
    family="monospace",
)
ax.axis("off")
fig.tight_layout(pad=0)

def update(frame):
    overlay.set_data(movie[frame])
    time_text.set_text(f"t = {frame_times[frame]:7.3f} s")
    return overlay, time_text

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_times),
    blit=True,
)

writer = FFMpegWriter(
    fps=(line_rate_hz / lines_per_cycle) * PLAYBACK_RATE,
    codec="libx264",
    extra_args=["-pix_fmt", "yuv420p", "-crf", "18", "-movflags", "+faststart"],
)

animation.save(output_path, writer=writer)
plt.close(fig)

print("Saved:", output_path)
display(Video(str(output_path), embed=False, html_attributes="controls loop"))

## Interpretation

This is a cycle-resolved reconstruction of what SLAP2 actually measured:

- TIFF pixels retain their original raster geometry.
- Integration values retain their original superpixel geometry.
- Unsampled pixels are not inferred.
- Repeated integration samples within a cycle are averaged only to form the
  cycle-rate movie frame.
- `dff` mode is a visualization transform applied after raw reconstruction; it
  is not the output of the voltage extraction pipeline.

For maximum raw appearance, use `VALUE_MODE="raw"`. For clearer ASAP8 activity,
use `VALUE_MODE="dff"` and reduce `PLAYBACK_RATE` to `0.5`.